# Adaptive Robot Intelligence: image-based policy on a GPU (M15)

This notebook runs the repository's own scripts at full scale on a free Colab GPU.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

What it does:
1. Installs the project.
2. Trains the image-based student policy (two 64×64 cameras → actions) with a teacher that sees simulator state, then DAgger.
3. Evaluates on the fixed 100 episodes, with and without the moved-cube disturbance.
4. *(Optional)* Trains LeRobot's ACT policy on the same data.

What has been verified: every script below was run on CPU at small scale in the repository (see `docs/results_m15.md`). The full-scale GPU numbers are yours to produce; the times quoted here are estimates.

In [ ]:
BRANCH = "main"   # use "stage4-m15-vision-policy" if that branch is not merged yet
!git clone -q -b $BRANCH https://github.com/alper-sharip/adaptive-robot-intelligence.git
!git clone -q --depth 1 https://github.com/google-deepmind/mujoco_menagerie.git
%cd adaptive-robot-intelligence
!pip install -q -e .
import os
os.environ["MUJOCO_GL"] = "egl"          # off-screen rendering on the GPU
import torch
print("GPU available:", torch.cuda.is_available())

## 1. Sanity check (about 2 minutes)
The same tests that pass on CPU should pass here.

In [ ]:
!pip install -q pytest
!python -m pytest -q tests/test_learning.py tests/test_vision_policy.py

## 2. Train the image-based student
500 demonstrations, 30 epochs, 3 DAgger iterations. Rendering the demonstrations takes most of the first phase.
Expect roughly 30-60 minutes in total on a T4 (an estimate, not a measurement).

In [ ]:
!python scripts/train_vision.py --demos 500 --epochs 30 --dagger-iters 3 --dagger-episodes 100 --name vision_full

## 3. Evaluate on the fixed 100 episodes
Compare with `docs/results_m15.md` (CPU-scale run) and `docs/results_stage4.md` (state-based policy).

In [ ]:
!python scripts/evaluate_bc.py outputs/models/vision_full.pt
!python scripts/evaluate_bc.py --disturb outputs/models/vision_full.pt

In [ ]:
!python scripts/bc_demo.py outputs/models/vision_full.pt --seed 3 --disturb
from IPython.display import Image
Image(open("outputs/vision_full_seed3_disturbed.gif", "rb").read())

## 4. (Optional) ACT from LeRobot
ACT predicts a chunk of future actions with a transformer. The script was smoke-tested on CPU
(conversion, a few training steps, one rollout) against **lerobot 0.7.0**; a full training run has not been done,
so treat the result as an experiment. If a newer lerobot breaks the script, pin the version: `pip install "lerobot[dataset]==0.7.0"`.

In [ ]:
!pip install -q "lerobot[dataset]"
!python scripts/lerobot_act.py convert
!python scripts/lerobot_act.py train --steps 20000
!python scripts/lerobot_act.py eval --n 100

## 5. Keep your results
Download the trained model and the evaluation file, then commit them to your repository from your Mac.

In [ ]:
from google.colab import files
files.download("outputs/models/vision_full.pt")
files.download("outputs/eval/bc_results.json")